In [23]:
# Cell 1: Import libraries
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [24]:
# Cell 2: Load the source tables
patients = pd.read_csv("patients.csv")
admissions = pd.read_csv("admissions.csv")
diagnoses = pd.read_csv("diagnoses.csv")

print("Patients shape:", patients.shape)
print("Admissions shape:", admissions.shape)
print("Diagnoses shape:", diagnoses.shape)

Patients shape: (86400, 8)
Admissions shape: (120000, 17)
Diagnoses shape: (271341, 6)


In [25]:
# Cell 3: Merge patients and admissions
# The result remains at one row per admission.
df = admissions.merge(patients, on="patient_id", how="inner")
print("Merged data shape:", df.shape)

Merged data shape: (120000, 24)


In [26]:
# Cell 4: One-hot encode and aggregate diagnosis categories
# Grouping with max() keeps one row per admission despite multiple diagnoses.
diag_ohe = pd.get_dummies(
    diagnoses[["admission_id", "diag_category"]],
    columns=["diag_category"],
    prefix="diag",
    dtype=int,
)
diag_ohe = diag_ohe.groupby("admission_id", as_index=False).max()
df = df.merge(diag_ohe, on="admission_id", how="left")

print("Shape after diagnosis merge:", df.shape)

Shape after diagnosis merge: (120000, 35)


In [27]:
# Cell 5: Remove leakage columns and encode known categorical features
cols_to_drop = [
    "admission_id",
    "patient_id",
    "hospital_id",
    "admit_date",
    "discharge_date",
    "readmitted_7d",
]
target = "readmitted_30d"
df_model = df.drop(columns=cols_to_drop, errors="ignore")

df_model[target] = pd.to_numeric(df_model[target], errors="coerce")
df_model = df_model.dropna(subset=[target]).copy()

categorical_cols = [
    "gender",
    "state",
    "admit_type",
    "ward_type",
    "discharge_type",
    "insurance_type",
]
existing_categorical_cols = [
    column for column in categorical_cols if column in df_model.columns
]
df_model = pd.get_dummies(
    df_model,
    columns=existing_categorical_cols,
    drop_first=True,
    dtype=int,
)

print("Data shape after cleaning and encoding:", df_model.shape)

Data shape after cleaning and encoding: (120000, 49)


In [28]:
# Cell 6: Ensure all feature columns are numeric
# Numeric-looking strings are converted; unsupported UUID/string columns are dropped.
for column in df_model.columns:
    if column != target and df_model[column].dtype == "object":
        converted = pd.to_numeric(df_model[column], errors="coerce")
        if converted.notna().any():
            df_model[column] = converted
        else:
            df_model = df_model.drop(columns=column)

X = df_model.drop(columns=[target]).apply(pd.to_numeric, errors="coerce")
y = df_model[target].astype(int)

print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Feature matrix shape: (120000, 48)
Target distribution:
readmitted_30d
0    105790
1     14210
Name: count, dtype: int64


In [29]:
# Cell 7: Split the data and fit preprocessing on training data only
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train))
X_test = scaler.transform(imputer.transform(X_test))

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (96000, 48)
Testing features: (24000, 48)


In [30]:
# Cell 8: Train and evaluate the Logistic Regression model
model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

C:\Users\AAKARSH\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


ROC-AUC: 0.7773

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94     21158
           1       0.61      0.09      0.16      2842

    accuracy                           0.89     24000
   macro avg       0.75      0.54      0.55     24000
weighted avg       0.86      0.89      0.85     24000

